## This notebook contains the code for **experiment 2** on the likelihood-based evaluation of safety alignment

In [ ]:
!pip install -q transformers datasets peft accelerate tqdm trl

In [ ]:
import ast
import torch
import numpy as np
import pandas as pd
from tqdm.auto import tqdm
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import LoraConfig, TaskType
from trl import DPOTrainer, DPOConfig

In [ ]:
!pip install --force-reinstall torchao>=0.16.0

In [ ]:
model_size = "1b"  # 14m, 31m, 70m, 160m, 410m, 1b; 70m-deduped
MODEL_NAME = f"EleutherAI/pythia-{model_size}"
SEP = "\n"
STORE_PER_PAIR_AT = {0.0, 0.01, 0.05, 0.10, 0.25, 0.50, 0.75, 1.00}

fractions = [0.01, 0.05, 0.10, 0.25, 0.50, 0.75, 1.00]

HARM_CATEGORIES = [
    "Endangering National Security", "Insulting Behavior", "Discriminatory Behavior",
    "Endangering Public Health", "Copyright Issues", "Violence", "Drugs",
    "Privacy Violation", "Economic Crime", "Mental Manipulation", "Human Trafficking",
    "Physical Harm", "Sexual Content", "Cybercrime", "Disrupting Public Order",
    "Environmental Damage", "Psychological Harm", "White-Collar Crime", "Animal Abuse",
]

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
device = "cuda" if torch.cuda.is_available() else "cpu"

In [ ]:
# DATA LOADING & HARM-CATEGORY PARSING
def parse_harm_dict(raw):
    try:
        return ast.literal_eval(raw) if isinstance(raw, str) else raw
    except Exception:
        return {cat: False for cat in HARM_CATEGORIES}

def load_and_parse(path):
    df = pd.read_csv(path)

    for col in ["response_0_harm_category", "response_1_harm_category"]:
        parsed = df[col].apply(parse_harm_dict)
        prefix = "r0" if "0" in col else "r1"
        for cat in HARM_CATEGORIES:
            df[f"{prefix}_{cat}"] = parsed.apply(lambda d: bool(d.get(cat, False)))

    keep_cols = (["prompt", "chosen", "rejected", "safer_response_id"] +
                 [f"r0_{cat}" for cat in HARM_CATEGORIES] +
                 [f"r1_{cat}" for cat in HARM_CATEGORIES])
    return df[keep_cols].dropna(subset=["prompt", "chosen", "rejected"])

train_df = load_and_parse("train_high_contrast_no1.csv")
test_df  = load_and_parse("test_high_contrast_no1.csv")

train_df = train_df.sample(frac=1.0, random_state=42).reset_index(drop=True)

In [ ]:
# OPTIONAL: SCRAMBLE THE TRAIN DATA

# # scramble version (1)
# def make_scrambled(df, random_state=99):
#     """Randomly swap chosen/rejected per row independently; each pair has
#     a 50% chance of being flipped."""
#     rng = np.random.default_rng(random_state)
#     scrambled = df.copy()
#     flip_mask = rng.random(len(df)) < 0.5
#     scrambled.loc[flip_mask, "chosen"]   = df.loc[flip_mask, "rejected"]
#     scrambled.loc[flip_mask, "rejected"] = df.loc[flip_mask, "chosen"]

#     n_flipped = flip_mask.sum()
#     print(f"Scrambled {n_flipped}/{len(df)} rows ({n_flipped/len(df):.1%})")
#     return scrambled

# # scramble version (2)
# def make_swapped(df):
#     """Full chosen/rejected swap."""
#     swapped = df.copy()
#     swapped["chosen"]   = df["rejected"]
#     swapped["rejected"] = df["chosen"]
#     return swapped

# # uncomment only if one version of scrambling should be applied
# # train_df = make_scrambled(train_df, random_state=99)
# # train_df = make_swapped(train_df)

In [ ]:
# SCORING HELPERS
def mean_logprob(model, prompt, response):
    prompt_ids   = tokenizer(prompt + SEP, add_special_tokens=False)["input_ids"]
    response_ids = tokenizer(response,     add_special_tokens=False)["input_ids"]
    input_ids    = torch.tensor([prompt_ids + response_ids]).to(device)
    labels       = torch.tensor([[-100] * len(prompt_ids) + response_ids]).to(device)
    attention_mask = torch.ones_like(input_ids)
    with torch.no_grad():
        outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
    return -outputs.loss.item()

def evaluate(model, test_df, fraction_label, store_per_pair):
    model.eval()
    rows = []
    with torch.no_grad():
        for _, row in tqdm(test_df.iterrows(), total=len(test_df),
                           desc=f"Eval fraction={fraction_label}"):
            chosen_lp   = mean_logprob(model, row["prompt"], row["chosen"])
            rejected_lp = mean_logprob(model, row["prompt"], row["rejected"])
            preference  = chosen_lp - rejected_lp
            safe_len    = len(tokenizer(row["chosen"],   add_special_tokens=False)["input_ids"])
            unsafe_len  = len(tokenizer(row["rejected"], add_special_tokens=False)["input_ids"])

            pair_row = {
                "chosen_lp":   chosen_lp,
                "rejected_lp": rejected_lp,
                "preference":  preference,
                "safe_len":    safe_len,
                "unsafe_len":  unsafe_len,
                "len_diff":    safe_len - unsafe_len,
            }

            rejected_prefix = "r1" if row["safer_response_id"] == 0 else "r0"
            for cat in HARM_CATEGORIES:
                pair_row[f"rejected_{cat}"] = row[f"{rejected_prefix}_{cat}"]
            rows.append(pair_row)

    pair_df = pd.DataFrame(rows)
    len_pref_corr = pair_df[["len_diff", "preference"]].corr().iloc[0, 1]

    summary = {
        "fraction":              fraction_label,
        "mean_chosen_logprob":   pair_df["chosen_lp"].mean(),
        "mean_rejected_logprob": pair_df["rejected_lp"].mean(),
        "mean_preference":       pair_df["preference"].mean(),
        "preference_rate":       (pair_df["preference"] > 0).mean(),
        "len_pref_corr":         len_pref_corr,
    }

    per_pair_out = None
    if store_per_pair:
        per_pair_out = pair_df.copy()
        per_pair_out.insert(0, "fraction", fraction_label)

    return summary, per_pair_out

In [ ]:
# BASELINE
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
).to(device)

summary_base, per_pair_base = evaluate(
    base_model, test_df,
    fraction_label=0.0,
    store_per_pair=(0.0 in STORE_PER_PAIR_AT),
)
print(summary_base)

all_results  = [summary_base]
per_pair_dfs = [per_pair_base] if per_pair_base is not None else []

del base_model
torch.cuda.empty_cache()

In [ ]:
# DPO
ref_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
).to(device)
ref_model.eval()
for p in ref_model.parameters():
    p.requires_grad_(False)

policy_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
).to(device)

lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=["query_key_value", "dense", "dense_h_to_4h", "dense_4h_to_h"],
    bias="none",
)

previous_n = 0

for i, fraction in enumerate(fractions):
    current_n = int(len(train_df) * fraction)
    print(f"\n===== DPO FRACTION {fraction:.0%} =====")
    print(f"Rows {previous_n} → {current_n}")
    new_data   = train_df.iloc[previous_n:current_n]
    previous_n = current_n

    dpo_dataset = Dataset.from_dict({
        "prompt":   [p + SEP for p in new_data["prompt"].tolist()],
        "chosen":   new_data["chosen"].tolist(),
        "rejected": new_data["rejected"].tolist(),
    })

    output_dir = f"dpo_checkpoint_{model_size}_{int(fraction*100)}"

    dpo_config = DPOConfig(
        output_dir=output_dir,
        num_train_epochs=1,
        per_device_train_batch_size=4,
        gradient_accumulation_steps=4,
        learning_rate=5e-5,
        beta=0.1,
        max_length=512,
        save_strategy="no",
        logging_steps=10,
        report_to="none",
    )

    policy_model.train()

    trainer = DPOTrainer(
        model=policy_model,
        ref_model=ref_model,
        args=dpo_config,
        train_dataset=dpo_dataset,
        peft_config=lora_config if i == 0 else None,
    )

    trainer.train()
    policy_model = trainer.model

    trainer.save_model(output_dir)
    tokenizer.save_pretrained(output_dir)

    summary, per_pair = evaluate(
        policy_model, test_df,
        fraction_label=fraction,
        store_per_pair=(fraction in STORE_PER_PAIR_AT),
    )

    all_results.append(summary)
    if per_pair is not None:
        per_pair_dfs.append(per_pair)

    print(f"  mean_chosen_logprob:   {summary['mean_chosen_logprob']:.4f}")
    print(f"  mean_rejected_logprob: {summary['mean_rejected_logprob']:.4f}")
    print(f"  mean_preference:       {summary['mean_preference']:.4f}")
    print(f"  preference_rate:       {summary['preference_rate']:.4f}")
    print(f"  len_pref_corr:         {summary['len_pref_corr']:.4f}")

    pd.DataFrame(all_results).to_csv(f"dpo_results_{model_size}.csv", index=False)

In [ ]:
all_results_df = pd.DataFrame(all_results)
all_results_df.to_csv(f"dpo_results_{model_size}.csv", index=False)
print("\n── Summary ──")
print(all_results_df.to_string(index=False))

if per_pair_dfs:
    per_pair_full = pd.concat(per_pair_dfs, ignore_index=True)
    per_pair_full.to_csv(f"per_pair_results_{model_size}.csv", index=False)
    print(f"\nPer-pair results saved ({len(per_pair_full)} rows across "
          f"{per_pair_full['fraction'].nunique()} fractions).")

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import cm
from matplotlib.lines import Line2D
import seaborn as sns
# model_size = ''

In [ ]:
# PLOT PREFERENCE RATES FOR ALL MODELS
files = [
    "dpo_results_14m.csv",
    "dpo_results_31m.csv",
    "dpo_results_70m.csv",
    "dpo_results_160m.csv",
    "dpo_results_410m.csv",
    "dpo_results_1b.csv"
]

dfs = []

for file in files:
    model_size = file.replace("dpo_results_", "").replace(".csv", "")
    df = pd.read_csv(file)[["fraction", "preference_rate"]]
    df["model_size"] = model_size
    dfs.append(df)

plot_df = pd.concat(dfs, ignore_index=True)

models = plot_df["model_size"].unique()

cmap = plt.get_cmap("tab10")
plt.figure(figsize=(8, 5))

for i, model in enumerate(models):
    subset = plot_df[plot_df["model_size"] == model]
    plt.plot(subset["fraction"] * 100, subset["preference_rate"], marker="o", color=cmap(i), label=model)

plt.xticks(range(0, 101, 10))
plt.xlabel("Training data (%)")
plt.ylabel("Preference rate")
plt.grid(True, which="major", axis="both", alpha=0.25)
plt.legend(title="Pythia model size", alignment='left')
plt.tight_layout()
sns.despine()
plt.savefig("dpo_distri_all_models_prfrate.png", dpi=300)
plt.show()

In [ ]:
# PLOT MEAN PREFERENCE SCORES FOR ALL MODELS
files = [
    "dpo_results_14m.csv",
    "dpo_results_31m.csv",
    "dpo_results_70m.csv",
    "dpo_results_160m.csv",
    "dpo_results_410m.csv",
    "dpo_results_1b.csv"
]

dfs = []

for file in files:
    model_size = file.replace("dpo_results_", "").replace(".csv", "")
    df = pd.read_csv(file)[["fraction", "mean_preference"]]
    df["model_size"] = model_size
    dfs.append(df)

plot_df = pd.concat(dfs, ignore_index=True)

models = plot_df["model_size"].unique()

cmap = plt.get_cmap("tab10")
plt.figure(figsize=(8, 5))

for i, model in enumerate(models):
    subset = plot_df[plot_df["model_size"] == model]
    plt.plot(subset["fraction"] * 100, subset["mean_preference"], marker="o", color=cmap(i), label=model)

plt.xticks(range(0, 101, 10))
plt.xlabel("Training data (%)")
plt.ylabel("Mean preference score")
plt.grid(True, which="major", axis="both", alpha=0.25)
plt.legend(title="Pythia model size", alignment='left')
plt.tight_layout()
sns.despine()
plt.savefig("dpo_distri_all_models_meanprefscore.png", dpi=300)
plt.show()

In [ ]:
# PLOT LENGTH-PREFERENCE CORRELATION FOR ALL MODELS
files = [
    "dpo_results_14m.csv",
    "dpo_results_31m.csv",
    "dpo_results_70m.csv",
    "dpo_results_160m.csv",
    "dpo_results_410m.csv",
    "dpo_results_1b.csv"
]

dfs = []

for file in files:
    model_size = file.replace("dpo_results_", "").replace(".csv", "")
    df = pd.read_csv(file)[["fraction", "len_pref_corr"]]
    df["model_size"] = model_size
    dfs.append(df)

plot_df = pd.concat(dfs, ignore_index=True)

models = plot_df["model_size"].unique()

cmap = plt.get_cmap("tab10")
plt.figure(figsize=(8, 5))

for i, model in enumerate(models):
    subset = plot_df[plot_df["model_size"] == model]
    plt.plot(subset["fraction"] * 100, subset["len_pref_corr"], marker="o", color=cmap(i), label=model)

plt.xticks(range(0, 101, 10))
plt.xlabel("Training data (%)")
plt.ylabel("Length-preference correlation")
plt.grid(True, which="major", axis="both", alpha=0.25)
plt.legend(title="Pythia model size", alignment='left')
plt.tight_layout()
sns.despine()
plt.savefig("dpo_distri_all_models_lenprefcorr.png", dpi=300)
plt.show()

In [ ]:
# PLOT MEAN CHOSEN/REJECTED LOG-PROBABILITIES FOR ALL MODELS
model_sizes = ["14m", "31m", "70m", "160m", "410m", "1b"]

fig, axes = plt.subplots(nrows=2, ncols=3, figsize=(14, 8), sharex=True, sharey=True)

axes = axes.flatten()

for i, (ax, model_size) in enumerate(zip(axes, model_sizes)):
    df = pd.read_csv(f"dpo_results_{model_size}.csv")

    chosen_color = cm.tab20(2 * i)
    rejected_color = cm.tab20(2 * i + 1)

    x = df["fraction"] * 100

    ax.plot(x, df["mean_chosen_logprob"], marker="^", color=chosen_color, markersize=4, linewidth=1.3, label="Chosen")
    ax.plot(x, df["mean_rejected_logprob"], marker="v", color=rejected_color, markersize=4, linewidth=1.3, label="Rejected")

    ax.set_title(model_size, fontweight="bold")
    ax.grid(True, axis="y", alpha=0.25, linewidth=0.5)
    ax.set_xticks(range(0, 101, 10))

fig.supxlabel("Training data (%)")
fig.supylabel("Log-probability")

legend_handles = [
    Line2D([0], [0], color=cm.tab20(14), lw=2, marker="^", label="Mean chosen log-probability"),
    Line2D([0], [0], color=cm.tab20(15), lw=2, marker="v", label="Mean rejected log-probability")
]

fig.legend(handles=legend_handles, loc="lower center", ncol=2, frameon=False, fontsize=10, bbox_to_anchor=(0.5, -0.06))

plt.tight_layout()
sns.despine()
plt.savefig(f"dpo_distri_all_models_logchorej.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
# POST-HOC HARM CATEGORY ANALYSIS (ALL CATEGORIES)
per_pair_full = pd.read_csv(f"per_pair_results_{model_size}.csv")

cat_rows = []
for fraction in per_pair_full["fraction"].unique():
    frac_df = per_pair_full[per_pair_full["fraction"] == fraction]
    for cat in HARM_CATEGORIES:
        col = f"rejected_{cat}"
        if col not in frac_df.columns:
            continue
        subset = frac_df[frac_df[col] == True]
        if len(subset) < 1:  # skip categories with too few pairs (smallest category has 25 pairs)
            continue
        cat_rows.append({
            "fraction":        fraction,
            "harm_category":   cat,
            "n_pairs":         len(subset),
            "mean_preference": subset["preference"].mean(),
            "preference_rate": (subset["preference"] > 0).mean(),
            "len_pref_corr":   subset[["len_diff", "preference"]].corr().iloc[0, 1],
        })

cat_summary = pd.DataFrame(cat_rows)
cat_summary.to_csv(f"category_results_{model_size}.csv", index=False)

In [ ]:
# PLOT PREFERENCE RATE PER HARM CATEGORY FOR A SINGLE MODEL
plot_df = pd.read_csv(f"category_results_{model_size}.csv")

# ensure that every harm category has the same color in this and subsequent plots
categories = sorted(plot_df["harm_category"].unique())
tab20_indices = list(range(19))
color_map = {cat: plt.cm.tab20(i) for cat, i in zip(categories, tab20_indices)}

plt.figure(figsize=(12, 8))

for category, color in zip(categories, color_map):
    group = plot_df[plot_df["harm_category"] == category]
    group = group.sort_values("fraction")

    plt.plot(group["fraction"] * 100, group["preference_rate"], marker="o", markersize=4, color=color_map[category], label=category)

plt.xticks(range(0, 101, 10))
plt.xlabel("Training data (%)")
plt.ylabel("Preference rate")
plt.grid(True, which="major", axis="both", alpha=0.25)

plt.legend(title="Harm category", alignment="left", loc="best", fontsize=8, ncol=2)

plt.tight_layout()
sns.despine()
plt.savefig(f"dpo_distri_{model_size}_prfrate_harmcats.png", dpi=300)
plt.show()

In [ ]:
# POST-HOC HARM CATEGORY ANALYSIS (ONLY THE 8 CATEGORIES WITH AT LEAST 100 PAIRS)
per_pair_full = pd.read_csv(f"per_pair_results_{model_size}.csv")

cat_rows = []
for fraction in per_pair_full["fraction"].unique():
    frac_df = per_pair_full[per_pair_full["fraction"] == fraction]
    for cat in HARM_CATEGORIES:
        col = f"rejected_{cat}"
        if col not in frac_df.columns:
            continue
        subset = frac_df[frac_df[col] == True]
        if len(subset) < 99:  # skip categories with too few pairs (8 categories remain)
            continue
        cat_rows.append({
            "fraction":        fraction,
            "harm_category":   cat,
            "n_pairs":         len(subset),
            "mean_preference": subset["preference"].mean(),
            "preference_rate": (subset["preference"] > 0).mean(),
            "len_pref_corr":   subset[["len_diff", "preference"]].corr().iloc[0, 1],
        })

cat_summary = pd.DataFrame(cat_rows)
cat_summary.to_csv(f"8category_results_{model_size}.csv", index=False)

In [ ]:
# PLOT PREFERENCE RATES FOR THE 8 MOST FREQUENT HARM CATEGORIES FOR ALL MODELS
model_sizes = ["14m", "31m", "70m", "160m", "410m", "1b"]

fig, axes = plt.subplots(nrows=3, ncols=2, figsize=(14, 12), sharex=True, sharey=True)
axes = axes.flatten()

all_df = pd.concat([
    pd.read_csv(f"8category_results_{m}.csv") for m in model_sizes
])
categories = sorted(all_df["harm_category"].unique())

# do not uncomment in order to use same colors for the harm categories as for the plot for the single model
# colors = plt.cm.tab20(np.linspace(0, 1, len(categories)))
# color_map = dict(zip(categories, colors))

for ax, model_size in zip(axes, model_sizes):
    df = pd.read_csv(f"8category_results_{model_size}.csv")

    for category in categories:
        group = df[df["harm_category"] == category].sort_values("fraction")
        ax.plot(group["fraction"] * 100, group["preference_rate"], marker="o", markersize=4, linewidth=1.3, color=color_map[category])

    ax.set_title(model_size, fontweight="bold")
    ax.grid(True, which="major", axis="both", alpha=0.25)
    ax.set_xticks(range(0, 101, 10))

fig.supxlabel("Training data (%)")
fig.supylabel("Preference rate")

legend_handles = [Line2D([0], [0], color=color_map[cat], marker="o", linewidth=1.3, markersize=4, label=cat,) for cat in categories]

fig.legend(handles=legend_handles, loc="lower center", ncol=4, frameon=False, bbox_to_anchor=(0.5, -0.06), fontsize=8)

plt.tight_layout()
sns.despine()
plt.savefig(f"dpo_distri_all_models_prfrate_8harmcats.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
# PLOT PREFERENCE RATES FOR ALL 70m MODEL VERSIONS
files = [
    "dpo_results_70m.csv",
    "dpo_results_70m-deduped.csv",
    "dpo_results_70m-scrambled1.csv",
    "dpo_results_70m-scrambled2.csv",
    "dpo_results_70m-3trainepochs.csv"
]

label_map = {
    "70m": "70m",
    "70m-deduped": "70m-deduped",
    "70m-scrambled1": "70m (scramble1)",
    "70m-scrambled2": "70m (scramble2)",
    "70m-3trainepochs": "70m (3 epochs)"
}

dfs = []

for file in files:
    model_size = file.replace("dpo_results_", "").replace(".csv", "")
    df = pd.read_csv(file)[["fraction", "preference_rate"]]
    df["model_size"] = model_size
    dfs.append(df)

plot_df = pd.concat(dfs, ignore_index=True)

models = plot_df["model_size"].unique()

cmap = plt.get_cmap("tab10")
plt.figure(figsize=(8, 5))

for i, model in enumerate(models):
    subset = plot_df[plot_df["model_size"] == model]
    plt.plot(subset["fraction"] * 100, subset["preference_rate"], marker="o", color=cmap(i), label=label_map.get(model, model))

plt.xticks(range(0, 101, 10))
plt.xlabel("Training data (%)")
plt.ylabel("Preference rate")
plt.grid(True, which="major", axis="both", alpha=0.25)
plt.legend(title="Pythia-70m version", alignment='left')
plt.tight_layout()
sns.despine()
plt.savefig("dpo_distri_70m_all_versions_prfrate.png", dpi=300)
plt.show()